<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg", align="left">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.975 · Deep Learning · PAC1</p>
<p style="margin: 0; text-align:right;">2025-2 · Màster universitari en Ciència de dades (Data science)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicació</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>


<h1>PAC 1: Xarxes neuronals convolucionals amb Keras i Pytorch - Classificació d'imatges satel·litàries</h1>

<p>Al llarg d'aquesta pràctica implementarem diversos models de xarxes neuronals per classificar les imatges d'una base de dades d'imatges satel·litàries. Concretament es desenvoluparan les tasques següents:</p>

<ol>
  <li>Descàrrega, anàlisi i preprocessament de les dades (1.5 pts)</li>
  <li>Xarxa neuronal completament connectada (Dense NN) (1.5 pts)</li>
  <li>Xarxa convolucional (CNN) petita (2 pts)</li>
  <li>Autoencoders (2 pts)</li>
  <li>Transfer Learning amb models eficients: EfficientNetB0 (2 pts)</li>
  <li>Introducció a Pytorch: Replicant la nostra CNN (1 pt)</li>
</ol>

<p><u>Consideracions generals</u>:</p>

<ul>
  <li>La solució proposada no ha d'utilitzar mètodes, funcions o paràmetres declarats <strong><em>deprecated</em></strong> en versions futures.</li>
  <li>Aquesta PAC s'ha de fer de manera <strong>estrictament individual</strong>. Qualsevol indici de còpia serà penalitzat amb un suspens (D) per a totes les parts implicades i la possible evaluació negativa de la totalitat de l'assignatura.</li>
  <li>És necessari que l'estudiant indiqui <strong>totes les fonts</strong> que ha utilitzat per a la realització de la PAC. En cas contrari, es considerarà que l'alumne ha comès plagi, sent penalitzat amb un suspens (D) i la possible evaluació negativa de la totalitat de l'assignatura.</li>
  <li>Si s'utilitza qualsevol <strong>IA generativa</strong> en la resolució de la PAC <strong>s'ha de referenciar</strong> en aquelles seccions on s'ha utilitzat, com qualsevol altra font.</li>
</ul>

<p><u>Format del lliurament</u>:</p>

<ul>
  <li>Alguns exercicis poden suposar diversos minuts d'execució, de manera que el lliurament s'ha de fer en format <strong>Notebook</strong> i en format <strong>html</strong>, on es vegi el codi, els resultats i els comentaris de cada exercici. Podeu exportar el quadern a HTML a Jupyter Notebook des del menú File $\to$ Download as $\to$ HTML.</li>
  <li>Hi ha un tipus especial de cel·la per allotjar el text. Aquest tipus de cel·les serà molt útil per respondre a les diferents preguntes teòriques plantejades al llarg de l'activitat. Per canviar el tipus de cel·la a aquest tipus, al menú: Cell $\to$ Cell Type $\to$ Markdown.</li>
</ul>

<p><u>Dataset utilitzat</u>:</p>

<p>En aquesta PAC utilitzarem <strong>UC Merced Land Use Dataset</strong>, un conjunt de dades de lliure accés que consta d'imatges aèries d'alta resolució (2 m) d'una regió agrícola a la Vall Central de Califòrnia.</p>

<h2>0. Context i càrrega de llibreries</h2>

<p>Les imatges preses per satèl·lit són clau en la supervisió de l'ús i la cobertura del sòl, qüestions rellevants per a la gestió ambiental, la planificació urbana, la sostenibilitat i per combatre el canvi climàtic.</p>

<p>En aquesta pràctica, treballarem amb la base de dades <a href="https://www.kaggle.com/datasets/zeadomar/uc-mercedland">UC Merced Land Use Data</a>, que consisteix en imatges satel·litàries de 256x256 píxels de 21 escenes diferents: les classes són diverses, i contenen escenes i imatges d'avions o rius, entre altres categories.</p>

<p>Concretament treballarem amb una versió augmentada d'aquesta base de dades que està disponible en un <a href="https://www.kaggle.com/datasets/apollo2506/landuse-scene-classification">repositori de Kaggle</a>. En aquesta versió s'han dut a terme diversos processos d'augment de dades de tal manera que el nombre d'imatges per classe passa de 100 a 500.</p>

<p><strong>Nota: Es recomana realitzar la pràctica en l'entorn que ofereix la plataforma Kaggle, ja que ofereix un entorn gratuït amb 30 hores setmanals d'ús de GPU.</strong></p>

<p>Al llarg de tota la pràctica, per a la creació de les diferents xarxes, anirem alternant l'ús del model <a href="https://keras.io/guides/sequential_model/">Sequential</a> i el model <a href="https://keras.io/guides/functional_api/">Functional</a> de Keras a través de les classes <a href="https://keras.io/api/models/sequential/">Sequential</a> i <a href="https://keras.io/api/models/model/">Model</a> respectivament.</p>

<p>Comencem carregant les llibreries més rellevants:</p>

In [ ]:
# Importem tensorflow
import tensorflow as tf
print("TF version   : ", tf.__version__)

# Necessitarem GPU
print("GPU available: ", tf.config.list_physical_devices('GPU'))

# keras version is 2.11.0
import keras
print("Keras version   : ", keras.__version__)

In [ ]:
# Importem els elements de keras que utilizarem amb més freqüència
from keras.utils import image_dataset_from_directory
from keras.layers import (
    GlobalAveragePooling2D, Flatten,
    Dense, Dropout, Conv2D, Conv2DTranspose, BatchNormalization,
    MaxPooling2D, UpSampling2D, Rescaling, Resizing)
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from keras import Sequential, Model

In [ ]:
# Importem la resta de llibreries que necessitarem per a la PAC
import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

<h2>1. Descàrrega, anàlisi i preprocessament de les dades (1,5 punts)</h2>

<p>En aquest apartat explorarem la base de dades i prepararem la càrrega de les imatges per als models dels següents apartats.</p>

<p>Per a la descàrrega de la base de dades tenim 2 opcions depenent de si decidim treballar en local o des de l'entorn de Kaggle:</p>
<ul>
  <li>Si treballem en local, hem de descarregar la base de dades des del següent <a href="https://www.kaggle.com/datasets/apollo2506/landuse-scene-classification/download?datasetVersionNumber=3">enllaç</a> (és un arxiu .zip que ocupa 2 GB) i desconprimir-lo.</li>
  <li>Si treballem des de Kaggle, hem de pujar el Notebook de l'enunciat a la plataforma (per fer-ho podeu seguir els 6 primers passos del següent <a href="https://rajputankit22.medium.com/how-to-upload-my-own-notebook-to-kaggle-2b0dedbb5a6b">article</a>) i després, una vegada pujat el notebook, a la pestanya 'Input', clicar el botó '+ Add Input' i a la caixa de cerca introduir l'adreça 'https://www.kaggle.com/datasets/apollo2506/landuse-scene-classification'. Una vegada trobat el dataset, cal donar-li al botó '+' (Add Dataset), i des d'aquest moment ja tindreu accessible la base de dades a la ruta <code>../input/</code>.</li>
</ul>

<p>Una vegada tenim la base de dades accessible, anem a inspeccionar-la.</p>

<p>Les imatges es troben agrupades de 2 formes diferents:</p>
<ul>
  <li>A la carpeta <code>/landuse-scene-classification/images/</code> es troba el total de les imatges separades per classes (cada classe en una carpeta diferent). Però no s'ha realitzat una separació en conjunt d'entrenament i test (o entrenament, validació i test).</li>
  <li>A la carpeta <code>/landuse-scene-classification/images_train_test_val/</code> es troben 3 carpetes (<code>test</code>, <code>train</code> i <code>validation</code>) en les quals el total d'imatges s'ha separat de forma aleatòria. En cada una de les 3 carpetes, tenim imatges de les 21 classes agrupades en les seves corresponents carpetes. A la carpeta arrel <code>/landuse-scene-classification/</code> tenim 3 arxius .csv amb la distribució de cada carpeta.</li>
</ul>

<p>En aquesta pràctica utilitzarem el dataset ja particionat, és a dir, treballarem amb les imatges que es troben a la ruta <code>/landuse-scene-classification/images_train_test_val/</code>.</p>

<h3>1.1. Anàlisi dels arxius .csv</h3>

<p>A partir dels arxius .csv podem veure com s'han distribuït les dades. Per exemple:</p>

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/apollo2506/landuse-scene-classification/train.csv')

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [0,5 pts.]:</strong> A partir dels 3 arxius .csv es demana:
    <ul>
        <li>Extreure els noms de les 21 classes (això només cal fer-ho en un dels 3 arxius).</li>
        <li>Quantes instàncies tenim en total per a cada conjunt de dades?</li>
        <li>Comprovar que les classes estan balancejades en els 3 conjunts de dades (comptant, per a cada conjunt, quantes instàncies/exemples tenim per a cada classe).</li>
    </ul>        
</div>

In [ ]:
# Extraure noms de les classes


In [ ]:
# Nombre d'instàncies per conjunt


In [ ]:
# Nombre d'instàncies per classe


<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h3>1.2. Anàlisi de les carpetes d'imatges.</h3>

<p>Encara que se suposa que cada arxiu .csv reflecteix a la perfecció el contingut de cada conjunt de dades, no està de més assegurar-se que el contingut del mateix es correspon amb l'anotat en cada arxiu. Per a això es demana:</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [0,5 pts]:</strong> Proporciona, a partir de les carpetes d'imatges, el nombre d'imatges que tenim a cada categoria per a cada conjunt de dades, comprovant que coincideix amb el que estipula l'arxiu .csv, i visualitza a tall d'exemple una imatge per cada categoria. Quin rang dinàmic (valors mínim i màxim) tenen les imatges?
</div>

In [ ]:
# Per al conjunt de dades d'entrenament


In [ ]:
# Per al conjunt de dades de validació


In [ ]:
# Per al conjunt de dades de test


### 1.3. Creació dels conjunts de dades en format Keras/Tensorflow
​
Amb l'objectiu de crear una base de dades en el format Keras/Tensorflow a partir de les imatges proporcionades, utilitzarem la funció <code>**tf.keras.utils.image_dataset_from_directory()**</code>, ja que ens permet crear bases de dades a partir d'imatges guardades en carpetes.

<p>La documentació d'aquesta funció es troba tant a la web de <a href="https://keras.io/api/data_loading/image/">Keras</a> com a la de <a href="https://www.tensorflow.org/api_docs/python/tf/keras/utils/image_dataset_from_directory">Tensorflow</a>.</p>

<p>A més, aprofitarem per redimensionar les imatges i passar-les a mida 224x224, que és la mida amb la qual s'ha entrenat la xarxa EfficientNetB0 que utilitzarem en un apartat posterior.</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [0,5 pts]:</strong> Utilitza la funció <code>image_dataset_from_directory()</code> per generar 3 conjunts de dades (<code>train_data</code>, <code>val_data</code> i <code>test_data</code>) a partir de les carpetes analitzades. Les imatges han de ser redimensionades a mida 224x224 píxels RGB (224,224,3) i agrupades en lots de mida 32 (batch=32) mantenint el seu rang dinàmic.
</div>

<h2>2. Xarxa neuronal completament connectada (Dense NN) (1,5 punts)</h2>

<p>En aquest apartat, entrenarem i avaluarem un model molt senzill completament connectat per establir un resultat de referència.</p>

<p>Atès que en una xarxa neuronal artificial les entrades són unidimensionals, el primer que hem de fer és redimensionar les dades d'entrada (les imatges) per convertir-les en arrays d'una dimensió.</p>

<p>Com que treballar amb imatges de mida 224x224 en una xarxa completament connectada implicaria entrenar un nombre de paràmetres excessivament elevat, definirem un model en el qual es realitzarà prèviament un redimensionament de les imatges d'entrada a una mida de 32x32 i un aplanament (<em>flattening</em>) dels píxels per així generar un vector unidimensional de mida 3072 (32x32x3).</p>

<p>Posteriorment entrenarem un classificador (una xarxa completament connectada) per dur a terme la classificació de les nostres dades.</p>

<p>En aquest apartat utilitzarem les capes <a href="https://keras.io/api/layers/preprocessing_layers/image_preprocessing/resizing/">Resizing</a>, <a href="https://keras.io/api/layers/preprocessing_layers/image_preprocessing/rescaling/">Rescaling</a>, <a href="https://keras.io/api/layers/reshaping_layers/flatten/">Flatten</a>, <a href="https://keras.io/api/layers/core_layers/dense/">Dense</a> i <a href="https://keras.io/api/layers/regularization_layers/dropout/">Dropout</a> de Keras.</p>


<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
    <strong>Exercici:</strong> Implementa un model <strong>seqüencial</strong> de Keras (a partir de la classe <code>Sequential()</code>) amb les següents especificacions:
    <ul>
        <li>Una capa que redueixi les dimensions d'entrada de (224,224) a (32,32).</li>
        <li>Una capa de reescalat per aconseguir que els valors de la imatge estiguin entre 0 i 1.</li>
        <li>Una capa Flatten per convertir la imatge en un vector de 3072 posicions.</li>
        <li>Una capa completament connectada de 1024 neurones i activació ReLU.</li>
        <li>Una capa de Dropout (amb probabilitat 0.5).</li>
        <li>Una capa de sortida completament connectada corresponent a la classificació final, el nombre de neurones de la qual ha de ser igual al nombre de classes de la base de dades i amb la funció d'activació adequada per dur a terme aquesta tasca de classificació.</li>
    </ul>
        
Compilar i entrenar el model seguint les següents indicacions:
     <ul>
         <li>Utilitzar l'optimitzador Adam amb un <i>learning rate</i> de 0.0001.</li>
         <li>Entrenar durant 100 èpoques utilitzant <i>EarlyStopping</i> amb una persistència de 10 èpoques, monitoritzant la funció de pèrdua en el conjunt de validació, i guardant els pesos que millor resultat hagin obtingut.</li>
         <li>Monitoritzar la mètrica <i>accuracy</i> durant l'entrenament i la validació.</li>
         <li>Mostrar les gràfiques d'accuracy i loss. En cada gràfica s'ha de visualitzar la corba d'entrenament i la de validació. NOTA: Es recomana fer una funció que imprimeixi ambdues gràfiques per poder reutilitzar-la en pròxims apartats.</li>
         <li>Realitzar l'avaluació del model una vegada ha finalitzat l'entrenament per mostrar la pèrdua i l'exactitud final sobre les dades de test.</li>
    </ul>
    Preguntes a respondre: Quin és el nombre de paràmetres a entrenar? I el temps d'entrenament? Quina precisió s'obté amb aquest model? Comenta els resultats i les gràfiques d'entrenament.<br/>    
    <strong> NOTA: es recomana, al final de la creació de cada model, utilitzar la funció <code>summary()</code> per comprovar l'estructura de la xarxa creada, així com el nombre de paràmetres que s'han d'entrenar. Es recomana fer-ho en tots els exercicis.</strong>
</div>

In [ ]:
# Definició de la xarxa

In [ ]:
# Compilació de la xarxa


In [ ]:
# Entrenament de la xarxa


In [ ]:
# Plot del training loss i l'accuracy


<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h2>3. Xarxa convolucional petita (2 punts)</h2>

<p>Ateses les baixes prestacions del model anterior, provarem un altre tipus de xarxes amb l'objectiu d'obtenir uns millors resultats en la tasca de classificació que hem de dur a terme.</p>

<p>Les xarxes convolucionals (CNN) són especialment adequades per modelar dades on hi ha patrons en 2 dimensions, com és el cas de les imatges.</p>

<p>En la tasca de classificació, l'estructura d'una CNN es divideix en dos grans blocs:</p>

<ul>
    <li><strong>Bloc extractor de característiques</strong>: En aquest bloc es generen diferents nivells d'abstracció de la imatge d'entrada mitjançant capes convolucionals. Com més profundes són aquestes capes, més preparades estan per a la tasca de classificació.</li>
    <li><strong>Classificador</strong>: Aquest bloc està format per capes totalment connectades; la sortida d'aquest bloc serà la probabilitat associada a cada classe.</li>
</ul>

<p>En l'apartat anterior, el bloc "extractor de característiques" era extremadament simple, per no dir inexistent. En aquest apartat, farem ús de capes convolucionals per poder aprendre millors abstraccions de les imatges d'entrada per tal de millorar-ne la classificació.</p>

<p>En aquest apartat utilitzarem les capes <a href="https://keras.io/api/layers/convolution_layers/convolution2d/">Conv2D</a>, <a href="https://keras.io/api/layers/pooling_layers/max_pooling2d/">MaxPooling2D</a>, <a href="https://keras.io/api/layers/pooling_layers/global_average_pooling2d/">GlobalAveragePooling2D</a>, <a href="https://keras.io/api/layers/core_layers/dense/">Dense</a> i <a href="https://keras.io/api/layers/regularization_layers/dropout/">Dropout</a> de Keras.</p>

<p><strong>Nota: Es recomana, a partir d'aquest punt, realitzar l'entrenament en una màquina amb GPU (pot activar-se en plataformes com Google Colab o Kaggle) per tal de reduir els temps d'entrenament.</strong></p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
    <strong>Exercici [2 punts]:</strong> A partir del model <strong>funcional</strong> de Keras (i la classe <code>Model()</code>), implementa una xarxa amb les següents característiques:
    <ul>
        <li>Un bloc extractor de característiques que consti de:
            <ul>
                <li>Una capa d'entrada de dimensions adequades a les dades.</li>
                <li>Una capa de reescalat per aconseguir que els valors de la imatge estiguin entre 0 i 1.</li>
                <li>3 capes convolucionals amb mida de kernel (5x5) per a la primera i (3x3) per a les 2 següents. S'utilitzarà padding '<i>same</i>' i activació ReLU. El nombre de filtres per a cada capa convolucional serà 16, 32 i 64 respectivament.</li>
                <li>A cada capa convolucional la segueix una capa de <i>Max Pooling</i>.</li>
                <li>Una capa d'<i>average pooling</i> (GlobalAveragePooling2D) per reduir les dimensions a un vector de 64 dimensions.</li>
            </ul></li>
        <li>El classificador final segueix l'estructura del model de l'apartat anterior:
            <ul>
                <li>Una capa completament connectada de 1024 neurones i activació ReLU.</li>
                <li>Una capa de Dropout (amb probabilitat 0.5).</li>
                <li>Una capa de sortida completament connectada corresponent a la classificació final, el nombre de neurones de la qual ha de ser igual al nombre de classes de la base de dades i amb la funció d'activació adequada per dur a terme aquesta tasca de classificació.</li>
            </ul></li>
    </ul>
    
Compilar i entrenar el model seguint les següents indicacions:
     <ul>
         <li>Utilitzar l'optimitzador Adam amb un <i>learning rate</i> de 0.001.</li>
          <li>Entrenar durant 100 èpoques utilitzant <i>EarlyStopping</i> amb una persistència de 10 èpoques, monitoritzant la funció de pèrdua en el conjunt de validació, i guardant els pesos que millor resultat hagin obtingut.</li>
         <li>Monitoritzar la mètrica <i>accuracy</i> durant l'entrenament i la validació.</li>
         <li>Mostrar les gràfiques d'<i>accuracy</i> i <i>loss</i>. En cada gràfica s'ha de visualitzar la corba d'entrenament i la de validació.</li>
         <li>Realitzar l'avaluació del model una vegada ha finalitzat l'entrenament per mostrar la pèrdua i l'exactitud final sobre les dades de test.</li>
    </ul>
    Preguntes a respondre: Quin és el nombre de paràmetres a entrenar? I el temps d'entrenament? Quina precisió s'obté amb aquest model? Comenta els resultats i les gràfiques d'entrenament.
</div>

In [ ]:
# Definició de la xarxa

In [ ]:
# Compilació de la xarxa

In [ ]:
# Entrenament

In [ ]:
# Resultats

<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h2>4. Autoencoders (2 punts)</h2>

<p>En l'apartat anterior hem pogut observar que, utilitzant el tipus de xarxes adequat, podem obtenir millors resultats entrenant un nombre de paràmetres molt inferior. Això és degut al fet que les CNN aconsegueixen extreure les característiques principals de les dades proporcionades (imatges en el nostre cas).</p>

<p>En aquest apartat observarem aquesta capacitat des d'un altre punt de vista: el de <strong>codificar i decodificar una imatge</strong>.</p>

<p>Per a fer-ho, dissenyarem un autoencoder que sigui capaç de reduir la mida de les dades d'entrada però captant les característiques principals de les imatges per a poder dur a terme una bona reconstrucció d'aquestes.</p>

<p>Començarem reescalant externament les dades que utilitzarem, perquè estiguin en el rang (0,1), en lloc de fer-ho dins de la xarxa com hem fet en l'apartat anterior:</p>

In [ ]:
# data rescalling
normalization_layer = Rescaling(1./255)

normalized_train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
normalized_val_data = val_data.map(lambda x, y: (normalization_layer(x), y))

<p>
  A més, en un autoencoder, en lloc d'utilitzar les etiquetes com a objectiu (que és el que s'utilitza en un problema de classificació), han de ser les pròpies imatges les que s'utilitzin com a objectiu de la xarxa. Per tant, crearem una nova base de dades d'entrenament i validació on són les pròpies imatges les que facin d'etiquetes:
</p>

In [ ]:
train_data_auto = normalized_train_data.map(lambda x, y: (x, x))
val_data_auto = normalized_val_data.map(lambda x, y: (x, x))

Comprovem l'estructura de la nova base de dades:

In [ ]:
image_batch, label_batch = iter(train_data_auto).get_next()
print("Las dimensiones de un batch de imágenes es: {}".format(image_batch.shape))
print("Las dimensiones de un batch de etiquetas es: {}".format(label_batch.shape))

I que les dades tenen el rang dinàmic adequat:

In [ ]:
first_image = image_batch[0]
print("En la primera imagen los valores mínimo y máximo son {} y {}, respectivamente"
      .format(np.min(first_image),np.max(first_image)))

<h3>4.1. Disseny i entrenament de l'autoencoder</h3>

<p>Una vegada ja tenim les dades en el format adequat, dissenyarem l'autoencoder. Per a fer-ho, utilitzarem el bloc extractor de l'apartat anterior com a codificador (exceptuant la capa de GlobalAveragePooling, ja que necessitem preservar l'estructura espaial) i reflectirem la seva estructura en el descodificador utilitzant les capes <a href="https://keras.io/api/layers/convolution_layers/convolution2d_transpose/">Conv2DTranspose</a> i <a href="https://keras.io/api/layers/reshaping_layers/up_sampling2d/">UpSampling2D</a> de Keras.</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
    <strong>Exercici [1 punt]:</strong> A partir del model <strong>funcional</strong> de Keras (i la classe <code>Model()</code>), implementa un autoencoder amb les següents característiques:
    <ul>
        <li>El bloc codificador ha de tenir:
            <ul>
                <li>Una capa d'entrada de dimensions adequades a les dades.</li>
                <li>3 capes convolucionals amb mida de kernel (5x5) per a la primera i (3x3) per a les 2 següents. S'utilitzarà padding '<i>same</i>' i activació ReLU. El nombre de filtres per a cada capa convolucional serà 16, 32 i 64 respectivament.</li>
                <li>A cada capa convolucional la segueix una capa de <i>Max Pooling</i>.</li>
            </ul></li>
        <li>El bloc descodificador ha de tenir:
            <ul>
                <li>3 capes convolucionals amb mida de kernel (3x3) per a les 2 primeres i (5x5) per a l'última. S'utilitzarà padding '<i>same</i>' i activació ReLU. El nombre de filtres per a cada capa convolucional serà 64, 32 i 16, respectivament.</li>
                <li>A cada capa convolucional la segueix una capa d'<i>UpSampling2D</i>.</li>
                <li>Una última capa convolucional amb mida de kernel (3x3), amb 3 filtres i activació sigmoide.</li>
            </ul></li>
    </ul>
    
Compilar i entrenar el model seguint les següents indicacions:
     <ul>
         <li>Utilitzar l'optimitzador Adam amb un <i>learning rate</i> de 0.001.</li>
         <li>Utilitzar com a funció de pèrdua l'error quadràtic mitjà.</li>
         <li>Entrenar durant 100 èpoques utilitzant <i>EarlyStopping</i> amb una persistència de 10 èpoques, monitoritzant la funció de pèrdua en el conjunt de validació, i guardant els pesos que millor resultat hagin obtingut.</li>
         <li>Monitoritzar la pèrdua durant l'entrenament i la validació.</li>
         <li>Mostrar les gràfiques del <i>loss</i> (la corba d'entrenament i la de validació).</li>
    </ul>
</div>



In [ ]:
# Definició de la xarxa

In [ ]:
# Compilació de la xarxa

In [ ]:
# Entrenament de la xarxa

In [ ]:
# Representació del loss

<h3>4.2. Avaluació de l'autoencoder</h3>

<p>L'avaluació del model obtingut pot fer-se en aquest cas tant de forma quantitativa (calculant l'MSE entre les imatges originals i reconstruïdes del conjunt de test) com qualitativa (mostrant imatges originals i reconstruïdes).</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
    <strong>Exercici [1 punt]:</strong> Realitzar les següents operacions per avaluar les prestacions del model obtingut:
    <ul>
        <li>Partint del conjunt de test obtingut en el primer apartat de la pràctica:
            <ul>
                <li>Dur a terme el reescalat de les dades utilitzant la capa <code>normalization_layer</code> tal com s'ha fet amb els conjunts d'entrenament i test a l'inici d'aquest bloc.</li>
                <li>Generar el conjunt de dades <code>test_data_auto</code> en el qual les imatges siguin també l'objectiu i substitueixin les etiquetes. </li>
            </ul></li>
        <li>Realitzar l'avaluació del model una vegada ha finalitzat l'entrenament per mostrar la pèrdua final a partir de les dades de test.</li>
        <li>Imprimir per pantalla 4 parelles d'imatges (original i reconstruïda). Nota: a l'hora de representar les imatges correctament, recordeu que el seu rang dinàmic han de ser nombres enters entre 0 i 255.</li>
    </ul>
    Preguntes: Consideres que la reconstrucció és adequada? Quina <i>ràtio</i> de compressió s'aconsegueix amb aquest autoencoder? Considerem com a <i>ràtio</i> de compressió la relació entre la mida original de la imatge (224,224,3) i la de la representació més petita que arriba a fer el codificador (mida de la sortida de la seva última capa).
</div>

In [ ]:
# Normalització de les dades


In [ ]:
# Avaluació del model


In [ ]:
# Visualització de les dades



<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h2>5. Transfer learning amb models eficients: EfficientNetB0 (2 punts)</h2>

<p>Les xarxes neuronals convolucionals profundes ens ofereixen la possibilitat de millorar la capacitat d'aprenentatge d'un model. Arquitectures clàssiques com VGG16 en van ser pioneres, però avui dia existeixen famílies de models molt més optimitzades, com <strong>EfficientNet</strong>, que aconsegueixen millors precisions amb molts menys paràmetres.</p>

<h3>5.1. Transfer Learning</h3>

<p>En aquest apartat, aplicarem <a href="https://keras.io/guides/transfer_learning/">transfer learning</a> utilitzant el model <a href="https://keras.io/api/applications/efficientnet/#efficientnetb0-function">EfficientNetB0</a> preentrenat a <a href="http://www.image-net.org/">Imagenet</a>. L'adaptarem per classificar les 21 categories de la nostra base de dades. Un gran avantatge d'EfficientNet a Keras és que <strong>ja inclou la capa de preprocessament (reescalat) internament</strong>, per la qual cosa podem passar-li directament les nostres imatges en el rang [0, 255].</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [1 punt]:</strong> Implementa una xarxa seguint els següents passos:
    <ul>
        <li>Partir del model EfficientNetB0 amb els pesos entrenats a Imagenet (sense la part superior de classificació, <code>include_top=False</code>) i congelar-los.</li>
        <li>Afegir una capa <code>GlobalAveragePooling2D</code> a la sortida del model base.</li>
        <li>Afegir una capa densa de 50 neurones amb activació ReLU, seguida de la capa de sortida amb el nombre de neurones adequat per dur a terme la classificació i la funció d'activació corresponent.</li>
    </ul>
Compilar i entrenar el model seguint les següents indicacions:
     <ul>
         <li>Utilitzar l'optimitzador Adam amb un <i>learning rate</i> de 0.0001.</li>
         <li>Entrenar durant 100 èpoques utilitzant <i>EarlyStopping</i> amb una persistència de 10 èpoques, monitoritzant l'<i>accuracy</i> en validació i guardant els millors pesos.</li>
         <li>Mostrar les gràfiques d'accuracy i loss (entrenament i validació).</li>
         <li>Avaluar el model final sobre les dades de test.</li>
    </ul>
    Preguntes a respondre: Quin és el nombre de paràmetres a entrenar? Quina precisió s'obté comparat amb la xarxa convolucional simple de l'apartat 3? Comenta els resultats i lles gràfiques d'entrenament.
</div>

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# Carregar el model base preentrenat


In [ ]:
# Definició de la xarxa

In [ ]:
# Compilació de la xarxa


In [ ]:
# Entrenament de la xarxa

In [ ]:
# Resultats

<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h3>5.2. Fine-tuning</h3>

<p>Per millorar els resultats del <i>transfer learning</i>, aplicarem <i>fine-tuning</i>, que consisteix a descongelar el model base i reentrenar la xarxa completa durant unes poques èpoques amb un <i>learning rate</i> molt petit per ajustar els pesos finament al nostre domini.</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [1 punt]:</strong> Tornar a compilar el model amb els següents canvis:
    <ul>
        <li>Descongelar els pesos del model EfficientNetB0 (<code>base_model_eff.trainable = True</code>).</li>
    </ul>
Compilar i entrenar el model seguint les següents indicacions:
     <ul>
         <li>Utilitzar l'optimitzador Adam amb un <i>learning rate</i> de 0.00001.</li>
         <li>Entrenar durant 10 èpoques.</li>
         <li>Mostrar les gràfiques i realitzar l'avaluació final sobre les dades de test.</li>
    </ul>
    Comenta els resultats globals del procés.
</div>

In [ ]:
# Definició del model

In [ ]:
# Compilació de la xarxa

In [ ]:
# Entrenament

In [ ]:
# Resultats

<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<h2>6. Introducció a PyTorch: Xarxa Convolucional Bàsica (1 punt)</h2>

<p>Fins ara hem utilitzat TensorFlow/Keras, que ens ofereixen una API d'alt nivell molt còmoda mitjançant els mètodes <code>.fit()</code> i <code>.evaluate()</code>. No obstant això, en el món de la investigació i en gran part de la indústria, <strong>PyTorch</strong> és el <i>framework</i> dominant a causa de la seva flexibilitat i al fet que permet un control més profund i "pythònic" del flux d'execució.</p>

<p>A PyTorch, nosaltres mateixos hem de definir com les dades passen per la xarxa (<code>forward pass</code>) i com iterem sobre els lots per actualitzar els pesos en el que anomenem el <strong>bucle d'entrenament</strong> (<i>training loop</i>).</p>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Exercici [1 punt]:</strong> Implementa una xarxa convolucional senzilla i entrena-la usant PyTorch per classificar la nostra base de dades.
    <ul>
        <li><strong>Càrrega de dades:</strong> Utilitza <code>torchvision.datasets.ImageFolder</code> i <code>DataLoader</code> per carregar les imatges d'entrenament des de la ruta <code>train_dir</code>. Aplica transformacions per redimensionar a 224x224 i convertir a Tensor (la qual cosa reescala automàticament a valors entre 0 i 1). Lots de mida 32.</li>
        <li><strong>Definició del model:</strong> Crea una classe que hereti de <code>nn.Module</code>. Ha de contenir:
            <ul>
                <li>Dues capes convolucionals (<code>nn.Conv2d</code>), cadascuna seguida d'una activació ReLU i un <i>Max Pooling</i> (<code>nn.MaxPool2d</code>).</li>
                <li>Una capa <code>nn.AdaptiveAvgPool2d((1, 1))</code> per col·lapsar les dimensions espacials.</li>
                <li>Una capa lineal de classificació (<code>nn.Linear</code>) amb sortida a 21 classes.</li>
            </ul>
        </li>
        <li><strong>Bucle d'entrenament:</strong> Defineix com a funció de pèrdua <code>nn.CrossEntropyLoss</code> i utilitza l'optimitzador Adam (lr=0.001). Escriu un bucle d'entrenament de <strong>5 èpoques</strong> que recorri el DataLoader d'entrenament i imprimeixi per pantalla la pèrdua (<i>loss</i>) de cada època.</li>
    </ul>
    Comenta breument les diferències que has notat en programar aquest model a PyTorch respecte a Keras.
</div>

<div style="background-color: #fcf2f2; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Comentaris:</strong>
<br><br>
(Afegeix els teus comentaris substituint aquest text)
<br><br>
</div>

<div style="background-color: #c3fac4; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>Fonts:</strong>
<br><br>
Posa aquí les fonts utilitzades en l'elaboració de la PAC (incloent eines d'IA generativa).
<br><br>
    <ul>
        <li><strong>Fonts utilitzades en l'exercici 1:</strong> (escriu aquí la teva resposta)</li>
        <li><strong>Fonts utilitzades en l'exercici 2:</strong> (escriu aquí la teva resposta)</li>
        <li><strong>Fonts utilitzades en l'exercici 3:</strong> (escriu aquí la teva resposta)</li>
        <li><strong>Fonts utilitzades en l'exercici 4:</strong> (escriu aquí la teva resposta)</li>
        <li><strong>Fonts utilitzades en l'exercici 5:</strong> (escriu aquí la teva resposta)</li>
        <li><strong>Fonts utilitzades en l'exercici 6:</strong> (escriu aquí la teva resposta)</li>        
    </ul>
</div>